# Report — reconstructing CAMUS echocardiography with a VAE

**IAV — INSA Lyon**
Tayeb and Paul

This report follows seven experiments on the same question: can a variational autoencoder rebuild a 64×64 ultrasound frame, and can one axis of its latent code follow the size of the left ventricle?

The images come from CAMUS (500 patients, two views, end-diastole and end-systole). The network is a MONAI `AutoencoderKL`. Every number below is copied from a finished run: the printed test tables of experiments 1 to 3, and `artifacts/results_exp4.json`, `results_exp5.json`, `results_exp6.json`, `results_exp7.json` for the GPU runs. A ± value is the standard deviation across the 200 held-out test frames, not a confidence interval on the mean.

The run we keep is **experiment 5**. On the test patients it reaches MSE 0.00268, SSIM 0.808, PSNR 26.04 dB, and Spearman 0.933 between the first latent axis and the left-ventricle cavity area. Experiment 6 was larger and did worse on every score. Experiment 7 went back to the experiment 5 network and did align three latent axes (Spearman 0.918, 0.849 and 0.702), but the image fell back to 22.47 dB. Both results stay in the report.

The four numbers we use all the time are:

$$
\mathrm{MSE} = \frac{1}{N}\sum_i (\hat{x}_i - x_i)^2,
\qquad
\mathrm{MAE} = \frac{1}{N}\sum_i |\hat{x}_i - x_i|
$$

$$
\mathrm{PSNR} = 10 \log_{10}\frac{1}{\mathrm{MSE}}
\quad\text{(images already scaled to }[0,1]\text{)}
$$

$$
\mathrm{SSIM}(x,\hat{x})
\quad\text{compares local means, variances, and covariance (window 7)}
$$

MSE and MAE judge pixels one by one. PSNR is the same information as MSE, written in decibels, so a small MSE change is easier to read. SSIM asks whether the local structure survived: a blurry heart can have a decent MSE and a poor SSIM. We report both.


## Contents

1. [Use of AI](#use-of-ai)
2. [Question](#question)
3. [Data](#data)
4. [What the network is doing](#model)
5. [How we score a run](#metrics)
6. [What stayed fixed](#protocol)
7. [The seven experiments](#experiments)
8. [Comparison](#comparison)
9. [What we keep](#decision)
10. [Bibliography](#bibliography)


<a id="use-of-ai"></a>

## 1. Use of AI

An AI coding assistant was used while building the later notebooks and this report.

**What the assistant produced**

- Drafts of the training loops from experiment 4 onward: the weighted loss, the gradient term, the three-attribute loss of experiment 6, the exponential moving average, and this report notebook.
- The experiment 7 notebook, copied from the experiment 5 template, with a 12-channel code, a multi-scale SSIM, separate attribute weights, a gradient clip of 3, and the same exponential moving average. The scores below are the GPU run in `artifacts/results_exp7.json`.
- A first training run of experiment 6 that did not learn. The decoder started outside [0, 1], a hard clamp killed the gradient, and the validation loss did not move. The assistant replaced the hard clamp by a straight-through clamp (the loss still sees values in [0, 1], the gradient still flows) and the run was started again.

**What we decided, and what we check**

- The patient split, the seed, and the decision to decode the posterior mean μ at test time.
- The order of the experiments: first make a VAE that reconstructs, then make one latent axis follow the cavity, then spend capacity on sharpness.
- Every score in the tables. They are copied from the experiment outputs. They are not estimated.

A number in this report comes from a full training run. If a run failed its own quality checks, the failure is written down. Experiment 6 failed them. Experiment 7 passed Spearman(z0) > 0.85 and failed the image checks against experiment 5.


<a id="question"></a>

## 2. Question

An ultrasound frame is a grey sector, a dark cavity, a bright wall, and a lot of speckle. Two frames of the same patient at end-diastole and end-systole differ mostly by the size of the cavity. We want a model that does two jobs at once.

1. **Rebuild the frame.** The reconstruction \(\hat{x}\) should be close to the original \(x\), including the wall and the speckle, not only a smooth blob in the right place.
2. **Put the cavity size on one axis.** If we read the first latent channel and it ranks the frames in the same order as the left-ventricle area, we can trust that axis as a simple cardiac attribute. The other channels stay free for everything else (view, texture, the rest of the anatomy).

The practical question of the project is narrower: **which of the six training setups is best on patients the network never saw, when both jobs count?**

We do not chase the lowest loss on the training set. A model can memorize 1,600 frames and still fail on the next patient. The test set is 50 patients held out from the start, and it is read after the checkpoint has already been chosen on validation.


<a id="data"></a>

## 3. Data

CAMUS gives, for each patient, a 2-chamber view and a 4-chamber view, each at end-diastole (ED) and end-systole (ES). That is four frames per patient. The course images are already 64×64. An expert mask comes with each frame:

| Label | Structure |
| --- | --- |
| 0 | background |
| 1 | left-ventricle cavity |
| 2 | myocardium |
| 3 | left atrium |

Intensities above 1 are divided by 255, so every frame lives in [0, 1]. We do not divide by the maximum of each image. That would brighten a dark exam and make the loss incomparable across patients.

The split is by **patient**, with seed 12345, and it is the same split in every experiment. A frame from patient 12 never appears in two sets.

| Set | Patients | Frames | Role |
| --- | --- | --- | --- |
| Train | 400 | 1,600 | weight updates |
| Validation | 50 | 200 | choose the epoch |
| Test | 50 | 200 | read once, after the checkpoint is frozen |

The cavity is larger at ED than at ES on the training set (mean cavity fraction about 0.113 against 0.072). That check is there so a bug in the masks would be caught before any training. The attribute we supervise is this fraction, not a length and not the ejection fraction.

About one pixel in five belongs to the heart (`label > 0`). A loss that treats every pixel equally spends most of its effort on the black background. From experiment 4 on, heart pixels are weighted more than background pixels.


<a id="model"></a>

## 4. What the network is doing

### 4.1 A small code for a 64×64 image

The encoder is a stack of convolutions and residual blocks. Three stride-2 downsamplings take the image from 64×64 to 8×8. At that grid the encoder predicts two maps, μ and σ, with \(C\) channels. The code is the sample

$$
z = \mu + \varepsilon \sigma, \qquad \varepsilon \sim \mathcal{N}(0, I).
$$

The decoder turns \(z\) back into a 64×64 image. At **test** time we do not sample. We decode μ. Sampling would add a different noise every time we measure MSE, and the number would not be the reconstruction the model actually believes. MONAI's `forward` samples even in `eval` mode, so the notebooks call `encode`, then `decode(μ)`, on purpose.

A 4×8×8 code has 256 numbers for 4,096 pixels (compression ×16). An 8×8×8 code has 512 numbers (×8). A 16×8×8 code has 1,024 numbers (×4). A 12×8×8 code, the one in experiment 7, has 768 numbers (×5.33). A shorter code forces the network to throw information away. That is useful if we want a clean latent space, and harmful if we want the speckle back.

### 4.2 Why there is a KL term

Without a penalty, μ and σ can do whatever makes reconstruction easy. Often σ collapses toward 0 and the codes of two similar hearts sit far apart, with empty space between them. Interpolating between end-diastole and end-systole then walks through codes the decoder has never seen, and the frame breaks.

The KL term pulls every posterior toward the standard normal:

$$
\mathrm{KL} = \frac{1}{2}\sum_k \left(\mu_k^2 + \sigma_k^2 - \log \sigma_k^2 - 1\right).
$$

The training loss is reconstruction plus \(\beta\) times this KL. β is the compromise. In experiment 1, β = 0 still samples \(z\) but does not penalize the KL: validation PSNR is 21.06 dB and the KL stays around 1,000. β = 10⁻⁴ costs about 0.5 dB and brings the KL down to about 70. β = 10⁻³ costs about 3 dB and the reconstruction becomes a blur. We kept β = 10⁻⁴ there, because the assignment wants a continuous latent space and a reconstruction that has not collapsed.

Later experiments make β even smaller (down to 10⁻⁵) and warm it up from 0, so the network first learns to copy the image, then only gently tidies the latent space.

### 4.3 The attribute loss (from experiment 3)

We do not regress the cavity area. We only ask the **order** to match. Let \(z_0\) be the mean of the first latent channel over the 8×8 grid, and let \(a\) be the fraction of pixels with label 1. For every pair of frames in the batch:

$$
L_{\mathrm{attr}} = \mathrm{mean}_{i \neq j} \big| \tanh\big(\delta (z_{0,i} - z_{0,j})\big) - \mathrm{sign}(a_i - a_j) \big|.
$$

δ = 2. If frame \(i\) has the larger cavity, \(z_{0,i}\) should be larger. The loss does not care by how much, only by the direction. γ multiplies this term. γ = 10⁻³ (experiment 3) was too small: Spearman on the test set stayed at 0.023. γ = 0.05 (experiment 4) was enough to reach 0.943. Through experiment 5 the other channels are not in this loss. They can store the view, the speckle, and the rest of the mask. Experiment 6 applies the same ranking loss to \(z_0\), \(z_1\) and \(z_2\), then multiplies γ by the average of the three terms. Experiment 7 keeps the three axes and does not average: the weights are 0.030, 0.015 and 0.015. Channels 3 to 11 stay free.

Spearman correlation is the right score for this loss, because both the loss and Spearman care about rank, not about a linear slope.


<a id="metrics"></a>

## 5. How we score a run

All test metrics decode μ and compare it with the original frame in [0, 1].

| Metric | What a change means on these images |
| --- | --- |
| MSE | Average squared pixel error. 0.010 is a rough copy. 0.003 is a sharp one. Lower is better. |
| MAE | Average absolute error, easier to picture as a grey-level gap. Lower is better. |
| PSNR | Same ranking as MSE, in dB. Going from 21 dB to 26 dB is the gain from experiment 1 to experiment 5. Higher is better. |
| SSIM | Structure in a 7×7 window. 0.45 can mean a blurry but well-placed cavity. 0.81 means the walls and the speckle pattern are much closer. Higher is better. |
| Heart MSE | MSE restricted to `label > 0`. This is the number that matters clinically. The background is easy. |
| Heart PSNR | \(10\log_{10}(1/\mathrm{MSE})\) inside the mask, averaged over frames. Reported from experiment 5 on. |
| Spearman(z0, LV area) | Rank agreement between the first latent axis and the cavity fraction. Above 0.85 means the axis actually follows the ventricle. |

The checkpoint inside a run is **not** chosen on the test set. It is the epoch, after the KL warm-up, with the lowest validation reconstruction. The test set is touched after that choice.

Two metrics can disagree. Experiment 2 has a better MSE than experiment 1 (0.00765 against 0.01024) and a worse SSIM (0.445 against 0.622). The pixels are closer on average, and the local structure is worse. From experiment 4 on, the loss includes SSIM so that this trade-off is not left to chance.


<a id="protocol"></a>

## 6. What stayed fixed

These choices do not change between experiments, so a gain can be attributed to the thing we did change.

- Seed 12345, patient-wise 400 / 50 / 50 split, intensities in [0, 1].
- MONAI `AutoencoderKL`, 2D, one input channel, GroupNorm when the widths allow it.
- Adam or AdamW and cosine learning-rate decay. Gradient clipping is 1 from experiment 4 through experiment 6, and 3 in experiment 7.
- Test reconstruction from μ, not from a random sample.
- The same four frames per patient: `2CH_ED`, `2CH_ES`, `4CH_ED`, `4CH_ES`.

What we allowed ourselves to change, one experiment at a time: the width of the network, the length of the code, the loss, the KL weight β, the attribute weight γ (one shared weight, then one weight per axis), the gradient clip, an exponential moving average of the weights, the number of epochs, and the device (CPU for 1–3, an RTX 4060 Ti for 4–7).

Experiments 4 to 7 train in full FP32. TensorFloat-32 is off, because it would round the convolutions and the metric would move for a reason that is not the model.


<a id="experiments"></a>

## 7. The seven experiments

### 7.1 Experiment 1 — a first VAE, on CPU

Notebook: `experiments/cpu/exp_1_vae_camus.ipynb`.

The backbone is small: channels `(16, 32, 64, 64)`, one residual block per scale, latent **4×8×8** (256 coefficients), 473,649 parameters. The reconstruction term is an L1. β is chosen on validation only, over {0, 10⁻⁴, 10⁻³, 10⁻², 10⁻¹}, for 8 epochs each.

| β | Validation MAE | Validation PSNR | Validation SSIM | KL |
| --- | ---: | ---: | ---: | ---: |
| 0 | 0.0562 | 21.06 dB | 0.599 | 1041.65 |
| **10⁻⁴** | 0.0595 | 20.54 dB | 0.595 | 69.92 |
| 10⁻³ | 0.0811 | 17.87 dB | 0.472 | 0.95 |
| 10⁻² | 0.0795 | 18.00 dB | 0.494 | 0.01 |
| 10⁻¹ | 0.0780 | 18.08 dB | 0.507 | 0.00 |

The rule keeps the largest β that stays within 0.4 dB of the best positive-β PSNR. That is β = 10⁻⁴. At 10⁻³ the image is already soft: the prior has won and the heart has lost.

The selected VAE is then trained for 25 epochs. On the 200 test frames: **MSE 0.01024, MAE 0.057, PSNR 20.97 dB, SSIM 0.622, heart MSE 0.01511**. A mean-frame baseline sits at 17.75 dB, so the network does more than predict the average ultrasound sector. A deterministic autoencoder with the same backbone reaches 21.77 dB, and a VQ-VAE reaches 20.82 dB. The deterministic model is slightly sharper because it never spends capacity on the KL. The VAE pays that cost so that a walk between two codes stays on the ultrasound manifold. There is no attribute loss yet, so no Spearman is reported.

Training took 26.8 minutes on CPU. The best epoch was the last one (25), which usually means the model had not finished improving.

### 7.2 Experiment 2 — a wider code, still only MSE + KL

Notebook: `experiments/cpu/exp_2_vae_camus.ipynb`.

Channels become `(32, 64, 128, 128)`, latent **8×8×8** (512 coefficients), 1,888,449 parameters, one residual block per scale. The loss is MSE + β·KL with β fixed at 10⁻⁴, for 20 epochs.

Test: **MSE 0.00765, MAE 0.055, PSNR 21.70 dB, SSIM 0.445, heart MSE 0.01073**. Best epoch 19 of 20. Duration 30.5 minutes on CPU.

Pixel error improved. SSIM fell from 0.622 to 0.445. A wider MSE-only network can move the grey levels closer and still wash out the edges the SSIM window is looking at. This is why the later losses are not pure MSE.

### 7.3 Experiment 3 — the attribute axis appears, too weakly

Notebook: `experiments/cpu/exp_3_vae_camus.ipynb`.

The network goes back to a small backbone, channels `(16, 32, 64, 64)`, latent 8×8×8, 478,417 parameters, so that a CPU run stays short (12.6 minutes, 20 epochs). The new term is the pairwise ranking loss of section 4.3, with γ = 10⁻³ and δ = 2. β warms up linearly to 10⁻⁴ over 5 epochs.

Test: **MSE 0.01024, MAE 0.061, PSNR 20.84 dB, SSIM 0.703, heart MSE 0.01298, Spearman(z0, LV area) = 0.023**.

SSIM recovered, because the network is smaller and less free to blur in a way that only MSE rewards. The Spearman is indistinguishable from no relationship. γ = 10⁻³ does not compete with the reconstruction term: the axis is not trained, it is only named. The idea is right. The weight is not.

### 7.4 Experiment 4 — GPU, a weighted loss, and a real axis

Notebook: `experiments/gpu/exp_4_vae_camus.ipynb`. Device: RTX 4060 Ti.

Channels `(32, 64, 128, 128)`, two residual blocks per scale, latent 8×8×8, no attention, 3,256,641 parameters. The loss becomes

$$
L = 0.50\, L_{\mathrm{mse},w} + 0.25\, L_{1,w} + 0.25\,(1 - \mathrm{SSIM}) + \beta\, \mathrm{KL} + \gamma\, L_{\mathrm{attr}}.
$$

The weight map is \(W = 1 + 3\cdot\mathbf{1}(\mathrm{label}>0)\): a heart pixel counts as 4, a background pixel as 1. β warms up to 5×10⁻⁵ over 10 epochs. γ = 0.05. AdamW at 10⁻³, cosine down to 10⁻⁵, 80 epochs, full FP32.

Test: **MSE 0.00343 ± 0.00261, MAE 0.034 ± 0.010, PSNR 25.12 ± 1.70 dB, SSIM 0.787 ± 0.030, heart MSE 0.00518 ± 0.00206, Spearman 0.943**. Best epoch 76. Wall time 3.75 minutes, about 2.5 GB of VRAM.

This is the first run that looks like an ultrasound frame rather than a smooth approximation, and the first run whose latent axis actually tracks the cavity (Spearman 0.943, p about 10⁻⁹⁶). The quality checks written in the notebook all pass (MSE < 0.007, PSNR > 22 dB, SSIM > 0.72, heart MSE < 0.0095, Spearman > 0.85). The optional time budget of 120 seconds does not: the run took 225 seconds. We did not shrink the model to win that clock. The later experiments drop the time constraint on purpose.

### 7.5 Experiment 5 — the run we keep

Notebook: `experiments/gpu/exp_5_vae_camus.ipynb`.

Same latent grid, 8×8×8. The last stage widens from 128 to 256 channels, and a spatial attention block is added on the 8×8 grid, where the four cavities fit in one feature map. 8,233,153 parameters. The loss adds a weighted gradient term, so the endocardial and epicardial edges are not only a pixel penalty:

$$
L_{\mathrm{recon}} = 0.40\, L_{\mathrm{mse},w} + 0.25\, L_{1,w} + 0.25\,(1-\mathrm{SSIM}) + 0.10\, L_{\nabla,w}.
$$

\(W = 1 + 3.5\cdot\mathbf{1}(\mathrm{label}>0)\) (a heart pixel counts as 4.5). β warms up to 2×10⁻⁵ over 15 epochs, lower than in experiment 4, so the KL blurs less. γ = 0.03, slightly lower, so the attribute does not eat the texture channels. Only \(z_0\) is supervised, and only by the cavity (`label == 1`). AdamW at 10⁻³, cosine to 10⁻⁵, 100 epochs, batch 64, FP32.

The checkpoint is epoch 48, the lowest validation reconstruction after the warm-up. Training continued to epoch 100, and the validation curve did not beat epoch 48. Duration 6.3 minutes, peak VRAM about 3.0 GB.

Test, 200 frames:

| | Mean | Std |
| --- | ---: | ---: |
| MSE | 0.00268 | 0.00138 |
| Heart MSE | 0.00446 | 0.00159 |
| Background MSE | 0.00212 | 0.00149 |
| MAE | 0.0277 | 0.0061 |
| PSNR | 26.04 dB | 1.50 dB |
| Heart PSNR | 23.73 dB | 1.33 dB |
| SSIM | 0.808 | 0.022 |

The heart is harder than the background, which is what we expect: the wall is thin and the cavity border is where the error concentrates. Heart PSNR is about 2.3 dB below the full-image PSNR.

Latent axis, still only \(z_0\), on the test set:

| Attribute | In the loss? | Spearman with \(z_0\) |
| --- | --- | ---: |
| LV cavity (`label == 1`) | yes | **0.933** |
| Epicardial area (cavity + myocardium) | no | 0.882 |
| Left atrium (`label == 3`) | no | −0.221 |

The cavity axis is strong, and it also follows the epicardial area, because that area contains the cavity. The left atrium goes the other way (Spearman −0.221, p = 0.0016). That is not a bug in the metric. During the cardiac cycle the atrium and the ventricle often change in opposite directions, and nothing in the loss asked \(z_0\) to represent the atrium. It represents the ventricle, and the atrium comes along for free with the wrong sign.

Inside each view the cavity Spearman stays high (the notebook prints it for 2CH and for 4CH separately). The axis is not a disguised view label.

Experiment 5 beats experiment 4 on MSE, heart MSE, SSIM, and PSNR, and Spearman stays above 0.85 (0.933 against 0.943: a small giveaway, in exchange for a sharper image). This is the model we would submit.

### 7.6 Experiment 6 — more capacity, and a clear miss

Notebook: `experiments/gpu/exp_6_vae_camus.ipynb`.

The bet was that experiment 5 had plateaued at epoch 48 because the code was too small and the sampling noise was shaking the validation curve. So this run removes every limit we had been careful with.

- Channels `(64, 128, 256, 512)`, three residual blocks per scale, attention at 16×16 and at 8×8. **49,122,305 parameters.**
- Latent **16×8×8** (1,024 coefficients). Channels 0, 1, and 2 are supervised by the LV cavity, the myocardium (`label == 2`), and the left atrium. Channels 3 to 15 are free. Note that experiment 5's "epicardial area" was cavity plus myocardium. Here the second target is the myocardium alone.
- Loss: \(0.35\) weighted MSE, \(0.25\) weighted L1, \(0.25\) two-scale SSIM (64 and 32), \(0.10\) gradient L1, \(0.05\) L1 on the FFT amplitude. Heart weight \(W = 1 + 4\cdot\mathbf{1}(\mathrm{label}>0)\).
- β warms up to 10⁻⁵ over 20 epochs. γ = 0.025 is multiplied by the mean of the three attribute terms, so each axis receives about 0.008.
- 150 epochs, batch 32, AdamW at 8×10⁻⁴, cosine down to 5×10⁻⁶, gradient clip at 1.
- From epoch 20, an exponential moving average of the weights (decay 0.995) is the one evaluated and saved. Validation and test still decode μ.

A first launch did not train: at initialization about half of the decoder pixels fell outside [0, 1], the clamp zeroed their gradient, and the validation MSE stayed stuck. The loss still receives a clamped image, but the gradient is allowed through those saturated pixels. After that fix the run did train, for 39.3 minutes, peak VRAM 5.8 GB.

It is not a good model. The best EMA checkpoint is epoch 150, and the validation MSE is still 0.00533 there. The attribute term is still about 0.96, close to its value at initialization (a random ranking sits near 1). The axes never got organized.

Test:

| | Experiment 6 |
| --- | ---: |
| MSE | 0.00609 ± 0.00281 |
| Heart MSE | 0.00842 ± 0.00315 |
| MAE | 0.0422 ± 0.0094 |
| PSNR | 22.49 ± 1.63 dB |
| Heart PSNR | 21.00 ± 1.46 dB |
| SSIM | 0.682 ± 0.033 |
| Spearman(z0, LV cavity) | 0.156 |
| Spearman(z1, myocardium) | 0.018 |
| Spearman(z2, left atrium) | 0.220 |

The 3×3 Spearman matrix (rows \(z_0, z_1, z_2\), columns cavity, myocardium, atrium) has no dominant diagonal. The largest entry on the atrium column is 0.220, and it is not cleanly isolated on \(z_2\). Nothing in this matrix is a usable clinical axis.

Why the extra capacity did not help:

- The gradient clip at 1 is the same number as on the 8 M parameter model. On this 49 M parameter model the global gradient norm starts near 12, so every step is cut by about that factor. The learning rate looks like 8×10⁻⁴ and behaves like a much smaller one. At epoch 150 the learning rate has already fallen to 5×10⁻⁶ and the validation MSE is flat.
- The implemented attribute term is \(0.025 \times (L_{\mathrm{attr},0} + L_{\mathrm{attr},1} + L_{\mathrm{attr},2}) / 3\), about 0.008 per axis. That is too small next to a reconstruction loss that itself has trouble moving. The ranking term never leaves 0.96, so there is nothing for Spearman to pick up.
- A longer latent vector and a heavier decoder are only useful if the optimizer can actually use them. Here it could not, in the budget we set.

The targets we had written down (MSE < 0.0018, heart MSE < 0.003, SSIM > 0.85, PSNR > 27.5 dB, Spearman > 0.85 on the three axes) all fail. So do the simpler checks against experiment 5. We do not average experiment 6 with experiment 5, and we do not retune it on the test set.


### 7.7 Experiment 7 — the experiment 5 network, with the axes repaired

Notebook: `experiments/gpu/exp_7_vae_camus.ipynb`.

Experiment 6 missed for three reasons that show up in the training curve and in the loss, not only in the test table.

- 49.1 million parameters for 1,600 training images. The network was larger than the data.
- The gradient clip at 1 cut a global norm that started near 12, so the learning rate on the page was not the step the weights took.
- γ = 0.025 was applied to the average of the three ranking terms. Each axis saw about 0.008, and the ranking loss stayed near 1.

Experiment 7 does not add capacity. It copies the experiment 5 template and changes the pieces those three points named. The straight-through clamp from experiment 6 stays, so a saturated decoder pixel still receives a gradient.

- Channels `(32, 64, 128, 256)`, two residual blocks, attention only on the 8×8 grid. **8,251,841 parameters.**
- Latent **12×8×8** (768 coefficients, compression ×5.33). Channels 0, 1 and 2 are supervised by the LV cavity (`label == 1`), the myocardium (`label == 2`) and the left atrium (`label == 3`). Channels 3 to 11 are free.
- Reconstruction, with no FFT term: \(0.40\) weighted MSE, \(0.20\) weighted L1, \(0.25\) two-scale SSIM (64 and 32), \(0.15\) weighted gradient L1. Heart weight \(W = 1 + 4\cdot\mathbf{1}(\mathrm{label}>0)\).
- β warms up to \(1.5\times 10^{-5}\) over 15 epochs. The attribute weights are separate and are not averaged: 0.030 on \(z_0\), 0.015 on \(z_1\), 0.015 on \(z_2\).
- 120 epochs, batch 64, AdamW at \(10^{-3}\), cosine down to \(10^{-5}\), gradient clip at 3. An exponential moving average of the weights (decay 0.995) is copied at the end of epoch 14 and is the validation model from epoch 15. The checkpoint is the lowest EMA validation reconstruction after the warm-up.

The best EMA checkpoint is epoch 120, validation reconstruction 0.0746. Training took 7.8 minutes, peak VRAM about 3.0 GB, the same footprint as experiment 5. The curve is not smooth. At epoch 23 the EMA reconstruction is 0.0882. By epoch 38 it has jumped to 0.339. It then climbs back, and the last epoch is the one we keep. The validation MSE there is still 0.00497, above experiment 5.

Test, 200 frames:

| | Experiment 7 |
| --- | ---: |
| MSE | 0.00716 ± 0.01045 |
| Heart MSE | 0.00942 ± 0.01082 |
| Background MSE | 0.00646 ± 0.01054 |
| MAE | 0.044 ± 0.022 |
| PSNR | 22.47 ± 2.18 dB |
| Heart PSNR | 21.04 ± 2.01 dB |
| SSIM | 0.715 ± 0.046 |
| Spearman(z0, LV cavity) | 0.918 |
| Spearman(z1, myocardium) | 0.849 |
| Spearman(z2, left atrium) | 0.702 |

The image does not beat experiment 5, and it does not beat experiment 6 on MSE (0.00716 against 0.00609) or on heart MSE (0.00942 against 0.00842). PSNR is 22.47 dB, next to 22.49 dB for experiment 6 and 26.04 dB for experiment 5. SSIM is 0.715, above experiment 6 (0.682) and below experiment 5 (0.808). The checks against experiment 5 fail on MSE, heart MSE, SSIM and PSNR. Spearman(z0) > 0.85 passes.

The axes are the part that moved. Rows are \(z_0, z_1, z_2\). Columns are cavity, myocardium, atrium.

| | LV cavity | Myocardium | Left atrium |
| --- | ---: | ---: | ---: |
| \(z_0\) | 0.918 | 0.673 | −0.204 |
| \(z_1\) | 0.651 | 0.849 | −0.060 |
| \(z_2\) | −0.375 | −0.138 | 0.702 |

The diagonal is the strong entry of each row. Inside each view, Spearman(\(z_0\), cavity) stays high: 0.911 on 2CH and 0.924 on 4CH. The axis is not a disguised view label. Experiment 6's largest atrium entry was 0.220 and was not isolated on \(z_2\). Here \(z_2\) is the atrium axis, at 0.702. \(z_1\) is just under the 0.85 line we used for a single axis.

The three corrections did what they were aimed at for the latent code. They did not bring the image back. A smaller network, a looser clip, and a γ that is not divided by three are enough to rank the three areas. They are not enough, with this loss and these 120 epochs, to match the reconstruction of experiment 5. We do not average the runs, and we do not retune experiment 7 on the test set.


<a id="comparison"></a>

## 8. Comparison

Lower is better for MSE, MAE, and the durations. Higher is better for PSNR, SSIM, and Spearman. The best value of each row is in bold. A dash means the run did not measure that quantity. Spearman for experiments 4 and 5 is \(\mathrm{Spearman}(z_0, \text{LV area})\). Experiments 6 and 7 also supervise \(z_1\) and \(z_2\). On those two rows the bold value is the better of the two runs that measured them.

| | Exp 1 | Exp 2 | Exp 3 | Exp 4 | Exp 5 | Exp 6 | Exp 7 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Device | CPU | CPU | CPU | RTX 4060 Ti | RTX 4060 Ti | RTX 4060 Ti | RTX 4060 Ti |
| Latent | 4×8×8 | 8×8×8 | 8×8×8 | 8×8×8 | 8×8×8 | 16×8×8 | 12×8×8 |
| Parameters | 0.47 M | 1.89 M | 0.48 M | 3.26 M | 8.23 M | 49.1 M | 8.25 M |
| Epochs (best) | 25 (25) | 20 (19) | 20 (20) | 80 (76) | 100 (48) | 150 (150) | 120 (120) |
| Duration | 26.8 min | 30.5 min | 12.6 min | 3.8 min | 6.3 min | 39.3 min | 7.8 min |
| MSE | 0.01024 ± 0.01093 | 0.00765 ± 0.00538 | 0.01024 ± 0.01042 | 0.00343 ± 0.00261 | **0.00268 ± 0.00138** | 0.00609 ± 0.00281 | 0.00716 ± 0.01045 |
| Heart MSE | 0.01511 ± 0.01215 | 0.01073 ± 0.00486 | 0.01298 ± 0.01028 | 0.00518 ± 0.00206 | **0.00446 ± 0.00159** | 0.00842 ± 0.00315 | 0.00942 ± 0.01082 |
| MAE | 0.057 ± 0.026 | 0.055 ± 0.016 | 0.061 ± 0.026 | 0.034 ± 0.010 | **0.028 ± 0.006** | 0.042 ± 0.009 | 0.044 ± 0.022 |
| PSNR (dB) | 20.97 ± 2.68 | 21.70 ± 1.92 | 20.84 ± 2.48 | 25.12 ± 1.70 | **26.04 ± 1.50** | 22.49 ± 1.63 | 22.47 ± 2.18 |
| Heart PSNR (dB) | — | — | — | — | **23.73 ± 1.33** | 21.00 ± 1.46 | 21.04 ± 2.01 |
| SSIM | 0.622 ± 0.070 | 0.445 ± 0.065 | 0.703 ± 0.052 | 0.787 ± 0.030 | **0.808 ± 0.022** | 0.682 ± 0.033 | 0.715 ± 0.046 |
| Spearman(z0, LV) | — | — | 0.023 | **0.943** | 0.933 | 0.156 | 0.918 |
| Spearman(z1, myocardium) | — | — | — | — | — | 0.018 | **0.849** |
| Spearman(z2, left atrium) | — | — | — | — | — | 0.220 | **0.702** |

Reading the table as a story, not as a leaderboard:

- Experiments 1 to 3 are the CPU chapter. They establish the data, the β compromise, and the fact that an attribute loss with a tiny γ does nothing. The best CPU PSNR is 21.70 dB (experiment 2), with a poor SSIM.
- Experiment 4 is the jump. Moving to the GPU is not what improves the image: 80 epochs, a heart-weighted loss, and γ = 0.05 are. PSNR gains about 3.4 dB over experiment 2, and Spearman goes from absent to 0.943.
- Experiment 5 spends a bit of that Spearman (0.943 → 0.933) to gain 0.9 dB of PSNR and a lower heart MSE. The standard deviation of the MSE also shrinks (0.00261 → 0.00138), so the test patients are closer to each other. The model is more even, not only better on average.
- Experiment 6 undoes the jump. Its PSNR (22.49 dB) is near the CPU results, with six times more parameters than experiment 5 and six times the training time. Capacity was not the missing piece.
- Experiment 7 puts the experiment 5 network back, with a 12-channel code, a clip of 3, and a separate γ on each axis. The axes come back (0.918, 0.849, 0.702). The image does not: PSNR is 22.47 dB, the experiment 6 number, and the MSE standard deviation (0.01045) is back to the size of the CPU runs. Ranking three attributes and reconstructing the frame were not the same problem.

The standard deviations matter. Experiment 1's MSE std (0.01093) is as large as its mean: some test frames are fine and some are far off. Experiment 5's MSE std is 0.00138, about half the mean. The errors are smaller and more regular. Experiment 7's MSE std is 0.01045, larger than its mean of 0.00716, so the average hides a few bad frames again.


<a id="decision"></a>

## 9. What we keep

We keep **experiment 5**, checkpoint epoch 48, file `checkpoints/exp5_vae_best.pth`.

It is still the only run that is at once the best reconstruction (MSE, heart MSE, MAE, PSNR, SSIM) and has a latent axis we can explain (Spearman 0.933 with the cavity area). Experiment 4 remains the right model if the only goal is that one cavity axis, by a small margin (0.943 against 0.933). Experiment 7 is the right model if the goal is the myocardium axis or the atrium axis (0.849 and 0.702), and the wrong model if the goal is the image: its PSNR is 22.47 dB, about 3.6 dB below experiment 5, and its cavity Spearman (0.918) does not beat experiment 5 either. We would not ship experiment 4 or experiment 7 once experiment 5 exists, because the image is the point of the assignment and experiment 5 wins every image metric.

We do not keep experiment 6. A negative result is still a result: doubling the latent channels, doubling the width, adding a second attention scale, a spectral loss, and an exponential moving average did not beat a careful 8 M parameter model. The training curve says the optimizer was still at a high validation MSE when the learning rate had already annealed. The attribute term says the three axes were never ranked. Reporting the 49 M parameter network as the final model would hide both facts.

We do not keep experiment 7 either. The correction worked on the part experiment 6 had broken. The three ranking terms move, and the test matrix has a diagonal (0.918, 0.849, 0.702). The reconstruction did not follow. The checkpoint is the last epoch, with a validation reconstruction of 0.0746 against 0.0613 for experiment 5, and the test MSE is 0.00716. Reporting experiment 7 as the final model would trade a sharp ultrasound frame for three axes. That is the opposite of the priority set when experiment 5 was chosen.

What experiment 5 does not do, and what we do not pretend it does:

- It does not supervise the myocardium or the left atrium. Those Spearman numbers, 0.849 and 0.702, belong to experiment 7. On experiment 5 the atrium correlation of −0.221 is still a correlation, not a control knob.
- It does not segment. The mask is an input of the loss and of the metrics, not an output of the network.
- The test set was not used to choose β, γ, or the epoch. It was used, across the project, to decide which experiment to write up. That is one look at the test set per experiment, not a search over hyperparameters on the test patients.

Experiment 7 was the change we would have tried next: the experiment 5 backbone, three axes, and a γ large enough that the ranking term moves. It did move, on the validation curve and on the test set, and the image did not keep up. We stop on experiment 5. Another pass on experiment 7 would be a new hyperparameter search, and the test patients have already been scored.


<a id="bibliography"></a>

## 10. Bibliography

- S. Leclerc, E. Smistad, J. Pedrosa, A. Ostvik, et al. *Deep Learning for Segmentation using an Open Large-Scale Dataset in 2D Echocardiography.* IEEE Transactions on Medical Imaging, 38(9), 2198–2210, 2019. The CAMUS dataset.
- O. Bernard. *Assignment 2 — reconstructing CAMUS echocardiography with a VAE.* IAV, INSA Lyon. The course skeleton this project starts from.
- D. P. Kingma and M. Welling. *Auto-Encoding Variational Bayes.* ICLR, 2014. The ELBO, the reparameterization \(z = \mu + \varepsilon\sigma\), and the KL term used in every experiment.
- R. Rombach, A. Blattmann, D. Lorenz, P. Esser, and B. Ommer. *High-Resolution Image Synthesis with Latent Diffusion Models.* CVPR, 2022. The `AutoencoderKL` design MONAI implements.
- W. H. L. Pinaya et al. *Brain Imaging Generation with Latent Diffusion Models.* MICCAI Workshop, 2022. The same backbone on medical images.
- A. Pati and A. Lerch. *Attribute-based Regularization of Latent Spaces for Variational Auto-Encoders.* Neural Computing and Applications, 2021. The pairwise ranking loss behind γ from experiment 3 on.
- Z. Wang, A. C. Bovik, H. R. Sheikh, and E. P. Simoncelli. *Image Quality Assessment: From Error Visibility to Structural Similarity.* IEEE Transactions on Image Processing, 13(4), 2004. The SSIM reported throughout.
- I. Loshchilov and F. Hutter. *Decoupled Weight Decay Regularization.* ICLR, 2019. AdamW, used from experiment 4 on.
- MONAI, *AutoencoderKL* and *SSIMLoss*. https://docs.monai.io. Source of the architecture and of the SSIM implementation (window 7, data range 1).
